In [9]:
import pandas as pd
import numpy as np
import getpass
import os
import yaml
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Optional, Any, Tuple
import sys
import duckdb

sys.path.append('/Users/BKieft/Metabolomics/metatlas2')
import metatlas2.database_interact as dbi
import metatlas2.pubchem_retrieval as pcr
import metatlas2.load_tools as lt

In [10]:
# Required: Input file of compounds to create atlas from. Minimally, must have columns 'inchi_key', 'chromatography', 'polarity', 'rt_peak', 'mz', 'adduct'
ATLAS_INPUT = "/Users/BKieft/Metabolomics/metatlas2/data/atlas_tables/HILICZ/HILICZ_ISTD_POS.tsv"
ATLAS_NAME = "HILIC Positive ISTD Default"

# Optional: Atlas description
ATLAS_DESC = f"Default ISTD compounds for HILIC Positive"

In [ ]:
# Load configuration from YAML file
config_path = Path("/Users/BKieft/Metabolomics/metatlas2/configs/metatlas2_config.yaml")
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

METATLAS_DB_PATH = config["paths"]["main_database"]
ANALYST_NAME = getpass.getuser()
TIMESTAMP = datetime.now().isoformat()

## Load and Validate Input Data

In [12]:
atlas_compounds_df, chromatography, polarity = lt.load_atlas_input(ATLAS_INPUT)

Loading input data from: /Users/BKieft/Metabolomics/metatlas2/data/atlas_tables/HILICZ/HILICZ_ISTD_POS.tsv
    Detected chromatography: HILICZ
    Detected polarity: positive
    Loaded 65 rows from input table


In [ ]:
atlas_uid, atlas_associations_created = dbi.create_atlas_from_compounds(
    atlas_compounds_df,
    chromatography,
    polarity,
    METATLAS_DB_PATH,
    ATLAS_NAME,
    ATLAS_DESC,
    ANALYST_NAME,
    TIMESTAMP
)

Validating compounds exist in database: /Users/BKieft/Metabolomics/metatlas2/data/databases/metatlas.duckdb
Database contains 482 compounds, searching for missing compounds in input.
Compounds available for atlas: 65
Compounds missing from database: 0

All 65 compounds found in database!
Created new atlas: HILIC Positive ISTD Default (UID: atlas-e1c4aea2f54f447dbb37531b6dabbe6f). Adding compounds...

Atlas creation complete!
   Atlas Name: HILIC Positive ISTD Default
   Atlas UID: atlas-e1c4aea2f54f447dbb37531b6dabbe6f
   Atlas associations created: 65
   Method: HILICZ/positive


In [ ]:
dbi.validate_database(METATLAS_DB_PATH)


Database Validation:
   Compounds: 482
   RT/MZ References: 860
   Atlases: 2
   Atlas-Compound associations: 85
   Method combinations:
      HILICZ/positive: 404 references
      HILICZ/negative: 456 references
   Available atlases:
      atlas-e1c4aea2f54f447dbb37531b6dabbe6f
            HILIC Positive ISTD Default
            HILICZ positive
            65 compounds
            2025-08-27 16:13:19.302068
      atlas-c94e57c00d8245ebb72eef59c2862c8f
            HILIC Positive QC Default
            HILICZ positive
            20 compounds
            2025-08-27 16:12:40.092883


In [ ]:
#dbi.delete_atlas_from_db(db_path=METATLAS_DB_PATH, atlas_uid="atlas-8e545b4efa7545e6856be900f19714d3")